In [1]:
import wandb

In [34]:
api = wandb.Api()

In [50]:
entity = api.default_entity
project = api.projects(entity)[0].name
runs = api.runs(f"{entity}/{project}")

In [89]:
# from $run.config $df.columns
configs = ['tilt_beta', 'z_mix_ratio', 'tilt_temperature_start', 'seed', 'tilt_init_geom_ratio', 'tilt_candidate_multiplier']
columns = ['eval/reach_top_right/episode_reward_iqm', 'eval/reach_top_left/episode_reward_iqm',
           'eval/reach_bottom_right/episode_reward_iqm','eval/reach_bottom_left/episode_reward_iqm', 
           'eval/task_reward_iqm']

In [90]:
import pandas as pd

rows = []
log_interval = 20000

for i, run in enumerate(runs):
    if run.state == "running":
        continue
    
    df = run.history()
    ckpt = int(run.files()[0].name[:-7]) // log_interval

    row_dict = {}

    for config in configs:
        row_dict[config] = run.config[config]

    for column in columns:
        row_dict[column] = df[column].iloc[ckpt]
        # 또는 df.loc[ckpt, column] 가능, index가 0,1,2...이면

    rows.append(row_dict)

result_df = pd.DataFrame(rows)

In [91]:
result_df.to_csv('maze.csv')

In [ ]:
result_df.groupby(["z_mix_ratio", "tilt_temperature_start", 'tilt_init_geom_ratio', 'tilt_candidate_multiplier'])['eval/task_reward_iqm'].agg(["mean", "std", "count"])

<Runs chicioue512-seoul-national-university/<Project chicioue512-seoul-national-university/uncategorized>>